# T2 D08 Value Semantics - action-focused diagnostic v0.2

This experimental notebook leads with the decision a user must make: **raise an issue for RCA**, **review configuration**, **apply an expected semantic treatment**, or **accept the assessed data as following the expected path**. Tags remain evidence rather than automatic defect labels.

## Decision policy

| Diagnostic evidence | User decision | Meaning |
|---|---|---|
| `STALE_FROZEN`, populated `NOT_APPLICABLE`, or unclassified evidence | Raise issue for RCA | A source, transformation, domain, or treatment problem may exist. |
| Unresolved role binding or unscoped KB route | Review configuration | Coverage is incomplete; absence of a tag is not a pass. |
| Supported `CENSORED` or unpopulated `NOT_APPLICABLE` | Apply expected treatment | Handle correctly downstream; no RCA unless the declared context is wrong. |
| Assessed field with no triggered condition | Expected path | No diagnostic action is required. |

In [3]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown, HTML
pd.set_option('display.max_columns', 80)
pd.set_option('display.max_colwidth', 140)
cwd = Path.cwd().resolve()
EXPERIMENT_ROOT = next((p for p in [cwd, *cwd.parents] if (p / 'kb' / 'value_semantics_kb_v0_2.yaml').is_file()), None)
if EXPERIMENT_ROOT is None:
    raise RuntimeError('Run from the t2_d08_Value_semantics folder or one of its children.')
if str(EXPERIMENT_ROOT) not in sys.path:
    sys.path.insert(0, str(EXPERIMENT_ROOT))

# Source
INPUT_SOURCE = 'fixture'       # fixture or snapshot
FIXTURE = 'pd'                 # pd, lgd, ead
SNAPSHOT_ID = 'item_e243fa4e72f4'
TABLE_NAME = 'mr_pd_sample'
ELIGIBLE_ROLES = None          # e.g. {'feature', 'target', 'score'}
SELECTED_COLUMNS = None        # optional explicit column set
DICTIONARY_PATH = None         # optional external CSV/YAML dictionary
USE_AAR_SOURCED_DICTIONARY = True
USE_DICTIONARY_FOR_MATCHING = True
COMPARE_DICTIONARY_MODES = True

# Binding
USE_LLM_FALLBACK = True       # sends unresolved column metadata, never cell values
USE_REVIEWED_FIXTURE_BINDINGS = True
BINDING_OVERRIDES = {}
SNAPSHOT_RUNTIME_DECLARATIONS = {}
D11_ENV_FILE = EXPERIMENT_ROOT.parent / 't2_d11_dir_consistency' / '.env'
CHECKPOINT_PATH = EXPERIMENT_ROOT / '.runtime' / 'action_notebook_role_adjudication_v0_2.jsonl'
OUTPUT_ROOT = EXPERIMENT_ROOT / 'output' / 'action_focused_v0_2'

In [4]:
from functions.action_summary import build_action_focused_summary
from functions.cell_rule_execution import execute_value_semantics
from functions.column_binding_workflow import resolve_column_bindings
from functions.dictionary_io import load_dictionary, overlay_dictionary
from functions.kb_loader import load_terminology, load_value_semantics_kb
from functions.result_summary import summarize_value_semantics
from functions.role_matching import match_variable_to_roles, prepare_role_matcher
from functions.run_cell_diagnostic import export_execution, load_fixture
from functions.snapshot_input import dictionary_evidence_view, load_snapshot_input
kb = load_value_semantics_kb(EXPERIMENT_ROOT / 'kb' / 'value_semantics_kb_v0_2.yaml')
terminology = load_terminology(EXPERIMENT_ROOT / 'kb' / 'credit_risk_abbreviations_v0_3.yaml')
matcher = prepare_role_matcher(kb, terminology)

## 1. Load and scope the practical input

In [5]:
if INPUT_SOURCE == 'fixture':
    fixture = load_fixture(FIXTURE)
    data, full_dictionary = fixture['data'].copy(), fixture['dictionary'].copy()
    row_reference_column, declarations, source_label = 'ROW_ID', fixture['declarations'], FIXTURE.lower()
    aar_sourced_dictionary = None
else:
    snapshot = load_snapshot_input(
        snapshot_id=SNAPSHOT_ID, table=TABLE_NAME,
        include_dictionary_metadata=USE_AAR_SOURCED_DICTIONARY,
    )
    data, full_dictionary = snapshot['data'], snapshot['dictionary']
    aar_sourced_dictionary = snapshot['aar_sourced_dictionary']
    row_reference_column = snapshot['row_reference_column']
    declarations, source_label = SNAPSHOT_RUNTIME_DECLARATIONS, f'snapshot_{SNAPSHOT_ID}_{TABLE_NAME}'
if DICTIONARY_PATH:
    full_dictionary = overlay_dictionary(full_dictionary, load_dictionary(DICTIONARY_PATH))
selected_dictionary = full_dictionary.loc[~full_dictionary['column_name'].eq(row_reference_column)].copy()
if ELIGIBLE_ROLES:
    selected_dictionary = selected_dictionary.loc[selected_dictionary['role'].str.casefold().isin({r.casefold() for r in ELIGIBLE_ROLES})]
if SELECTED_COLUMNS:
    missing = set(SELECTED_COLUMNS) - set(full_dictionary['column_name'])
    if missing:
        raise KeyError(f'Selected columns absent from input table: {sorted(missing)}')
    selected_dictionary = selected_dictionary.loc[selected_dictionary['column_name'].isin(SELECTED_COLUMNS)]
selected_dictionary = selected_dictionary.reset_index(drop=True)
display(Markdown(f'**Input:** {source_label}; **shape:** {len(data):,} rows x {len(data.columns)} columns; **selected:** {len(selected_dictionary)} columns'))
if aar_sourced_dictionary is not None:
    aar_user_columns = aar_sourced_dictionary.loc[~aar_sourced_dictionary['column_name'].eq(row_reference_column)]
    description_count = int(aar_user_columns['description'].astype(str).str.strip().ne('').sum())
    display(Markdown(f'**AAR-sourced dictionary:** {len(aar_user_columns)} columns, {description_count} descriptions; enabled: **{USE_AAR_SOURCED_DICTIONARY}**'))
display(selected_dictionary[['column_name', 'data_type', 'role', 'description', 'sentinel_value']])

**Input:** pd; **shape:** 1,200 rows x 19 columns; **selected:** 18 columns

,column_name,data_type,role,description,sentinel_value
0,FACILITY_ID,string,identifier,Unique facility identifier,
1,RPT_QTR,string,period,Ordered quarterly reporting period,
2,PORTF_SEG,string,group,Portfolio monitoring segment,
3,CURR_DPD,integer,feature,Current contractual days past due,
4,INT_RT,float,feature,Current contractual interest rate,
5,RATE_TYPE,string,feature,Fixed or floating benchmark-linked pricing basis,
6,BRR,string,score,Internal borrower risk rating currently in force,
7,NOI,float,feature,Periodic net operating income,-999
8,OCC_PCT,float,feature,Current property occupancy percentage,
9,DFLT_12M,integer,target,Forward default outcome over the next twelve months,


## 2. Match columns and establish execution bindings

In [6]:
def retrieval_view(frame, mode):
    rows = []
    for column in frame.to_dict(orient='records'):
        result = match_variable_to_roles(column, matcher)
        rows.append({
            'column_name': column['column_name'], 'mode': mode, 'match_status': result['match_status'],
            'exact_role': result['exact_match']['role'] if result['exact_match'] else '',
            'top_candidates': ';'.join(x['role'] for x in result['detailed_candidates'][:5]),
        })
    return pd.DataFrame(rows)
with_dictionary = dictionary_evidence_view(selected_dictionary, include_dictionary_metadata=True)
without_dictionary = dictionary_evidence_view(selected_dictionary, include_dictionary_metadata=False)
retrieval_with, retrieval_without = retrieval_view(with_dictionary, 'with'), retrieval_view(without_dictionary, 'without')
dictionary_comparison = retrieval_with.merge(retrieval_without, on='column_name', suffixes=('_with', '_without'))
dictionary_comparison['candidate_set_changed'] = dictionary_comparison['top_candidates_with'] != dictionary_comparison['top_candidates_without']
if COMPARE_DICTIONARY_MODES:
    display(dictionary_comparison)
matching_dictionary = with_dictionary if USE_DICTIONARY_FOR_MATCHING else without_dictionary
adjudicator = None
if USE_LLM_FALLBACK:
    from functions.llm_role_adjudication import OpenAIRoleAdjudicator, configured_model, create_openai_client, load_adjudication_prompt
    adjudicator = OpenAIRoleAdjudicator(
        client=create_openai_client(D11_ENV_FILE),
        prompt=load_adjudication_prompt(EXPERIMENT_ROOT / 'prompts' / 'value_semantics_role_adjudication_v0_2.txt'),
        model=configured_model(D11_ENV_FILE),
    )
binding_results, inferred_bindings = resolve_column_bindings(
    matching_dictionary.to_dict(orient='records'), matcher, adjudicator=adjudicator,
    checkpoint_path=CHECKPOINT_PATH, binding_overrides=BINDING_OVERRIDES,
)
if INPUT_SOURCE == 'fixture' and USE_REVIEWED_FIXTURE_BINDINGS:
    selected_names = set(matching_dictionary['column_name'])
    execution_bindings = {name: roles for name, roles in fixture['bindings'].items() if name in selected_names}
    binding_authority = 'reviewed_fixture_bindings'
else:
    execution_bindings, binding_authority = inferred_bindings, 'exact_plus_llm_and_overrides'
display(binding_results)
print('Execution-ready bindings:', len(execution_bindings), 'of', len(selected_dictionary))

,column_name,mode_with,match_status_with,exact_role_with,top_candidates_with,mode_without,match_status_without,exact_role_without,top_candidates_without,candidate_set_changed
0,FACILITY_ID,with,exact_match,entity_id,,without,exact_match,entity_id,,False
1,RPT_QTR,with,candidate_match,,period;default_date;forward_labels;income_measure;internal_grade,without,candidate_match,,period;default_date;forward_labels;income_measure;internal_grade,False
2,PORTF_SEG,with,candidate_match,,segment;period;regime_stamp;priced_rate;internal_grade,without,no_lexical_evidence,,maturity_date;priced_rate;spread_over_benchmark;segment;internal_grade,True
3,CURR_DPD,with,exact_match,arrears_measure,,without,exact_match,arrears_measure,,False
4,INT_RT,with,candidate_match,,priced_rate;credit_limit;arrears_measure;amortisation_basis;spread_over_benchmark,without,no_lexical_evidence,,amortisation_basis;priced_rate;rate_basis;spread_over_benchmark;maturity_date,True
5,RATE_TYPE,with,exact_match,rate_basis,,without,exact_match,rate_basis,,False
6,BRR,with,exact_match,internal_grade,,without,exact_match,internal_grade,,False
7,NOI,with,exact_match,income_measure,,without,exact_match,income_measure,,False
8,OCC_PCT,with,candidate_match,,priced_rate;utilisation_measure;arrears_measure;internal_grade;credit_limit,without,candidate_match,,utilisation_measure;priced_rate;maturity_date;default_date;income_measure,True
9,DFLT_12M,with,candidate_match,,forward_labels;default_date;default_event;arrears_measure;outcome_state,without,candidate_match,,forward_labels;arrears_measure;priced_rate;maturity_date;default_date,True


,column_name,production_role,description_supplied,match_status,detailed_candidate_count,adjudication_used,checkpoint_hit,decision,direct_roles,resolved_roles,review_required,reason,response_id,error
0,FACILITY_ID,identifier,True,exact_match,0,False,False,MATCH,entity_id,entity_id,False,Deterministic exact match,,
1,RPT_QTR,period,True,candidate_match,12,True,False,MATCH,period,period,True,"'RPT_QTR' and business name 'Reporting quarter' clearly indicate an ordered reporting period / observation date for the row, matching th...",resp_09dd52677dff8797016a9e2637c5b88194b7c7dade58757e79,
2,PORTF_SEG,group,True,candidate_match,12,True,False,MATCH,segment,segment,True,Business name and description indicate a portfolio monitoring segmentation variable; allowed values are geographic labels (NORTH|SOUTH|E...,resp_01e40a50e712b4fd016a9e263a25d081978239bc2561e41ade,
3,CURR_DPD,feature,True,exact_match,0,False,False,MATCH,arrears_measure,arrears_measure,False,Deterministic exact match,,
4,INT_RT,feature,True,candidate_match,12,True,False,MATCH,priced_rate,priced_rate,True,"Business name and description both indicate the current contractual interest rate applied to the exposure, which matches priced_rate.",resp_06f27c04338a1be3016a9e263b770c8190a54a0c4beecc989a,
5,RATE_TYPE,feature,True,exact_match,0,False,False,MATCH,rate_basis,rate_basis,False,Deterministic exact match,,
6,BRR,score,True,exact_match,0,False,False,MATCH,internal_grade,internal_grade,False,Deterministic exact match,,
7,NOI,feature,True,exact_match,0,False,False,MATCH,income_measure,income_measure,False,Deterministic exact match,,
8,OCC_PCT,feature,True,candidate_match,12,True,False,MATCH,utilisation_measure,utilisation_measure,True,"Business name and description indicate a current occupancy percentage (0-1), which is a utilisation/occupancy measure rather than rate, ...",resp_0e1b5fc21a3b11d2016a9e263cf78881959ca178762f5e52ec,
9,DFLT_12M,target,True,candidate_match,12,True,False,MATCH,forward_labels,forward_labels,True,"The variable name and business name indicate a binary 12-month forward default target ('DFLT_12M', '12 month default'), and the descript...",resp_0024b97542ee35bb016a9e263e6778819382a188af5a1f9c28,


Execution-ready bindings: 18 of 18


## 3. Execute deterministic rules and build user actions

In [7]:
execution = execute_value_semantics(
    data, full_dictionary, execution_bindings, declarations, kb, row_reference_column=row_reference_column,
)
summaries = summarize_value_semantics(data, full_dictionary, execution_bindings, kb, execution)
actions = build_action_focused_summary(
    data, execution_bindings, execution, summaries, binding_results=binding_results,
)
executive = actions['executive_action_summary'].iloc[0]
decision_colors = {
    'RAISE_ISSUE_FOR_RCA': ('#FDE2E1', '#8B1E1E'),
    'REVIEW_CONFIGURATION': ('#FFF3CD', '#6B4F00'),
    'EXPECTED_PATH_APPLY_TREATMENT': ('#DDEBFF', '#184A78'),
    'EXPECTED_PATH_NO_ACTION': ('#DFF3E4', '#1D6334'),
}
background, foreground = decision_colors[executive['overall_user_decision']]
display(HTML(f"<div style='padding:16px;border-radius:8px;background:{background};color:{foreground};font-size:18px'><b>{executive['overall_user_decision']}</b><br>{executive['plain_language_outcome']}</div>"))
display(actions['executive_action_summary'])

,overall_user_decision,plain_language_outcome,raise_issue_items,configuration_review_items,expected_treatment_items,expected_no_action_fields,tagged_cells,important_note
0,RAISE_ISSUE_FOR_RCA,One or more findings warrant an issue and root-cause analysis; also complete any configuration reviews.,5,3,2,2,3550,"Counts indicate scope, not severity. Review the action queue and evidence before escalation."


## 4. What should the user do now?

Start with red/high RCA items, then resolve amber configuration gaps. Blue informational items describe required downstream treatment and do not, by themselves, warrant an issue.

In [8]:
queue = actions['action_queue']
def color_decision(value):
    colors = {
        'RAISE_ISSUE_FOR_RCA': 'background-color:#FDE2E1;color:#8B1E1E;font-weight:bold',
        'REVIEW_CONFIGURATION': 'background-color:#FFF3CD;color:#6B4F00;font-weight:bold',
        'APPLY_EXPECTED_TREATMENT': 'background-color:#DDEBFF;color:#184A78',
    }
    return colors.get(value, '')
if queue.empty:
    display(Markdown('**No action items were generated. Assessed data follows the expected path.**'))
else:
    display(queue.style.map(color_decision, subset=['user_decision']).format({'affected_share': '{:.1%}'}))

,priority,user_decision,scope,input_variable,matched_role,tag,reason_code,affected_cells,affected_rows,affected_share,sample_row_references,issue_statement,recommended_next_step,rca_warranted,evidence_reference
0,HIGH,RAISE_ISSUE_FOR_RCA,CELL_TAG,DEFAULT_FLAG,construction_constants,NOT_APPLICABLE,panel_scope_exclusion,1200,1200,100.0%,PD001_01;PD001_02;PD001_03;PD001_04;PD001_05,A value is populated where the KB says the field is not applicable for this row context.,"Raise a data-treatment issue. Determine whether the value is a placeholder, mapping leakage, or a valid policy exception before modelling use.",True,resolved_cell_tags
1,HIGH,RAISE_ISSUE_FOR_RCA,CELL_TAG,INT_RT,priced_rate,STALE_FROZEN,zero_variance_at_declared_grain,51,51,4.2%,PD001_05;PD001_06;PD001_07;PD005_05;PD005_06,The value did not change at the KB-declared monitoring grain despite sufficient observations.,Raise an RCA on the source refresh and transformation path; confirm whether the field is genuinely stable or its updates are frozen.,True,resolved_cell_tags
2,HIGH,RAISE_ISSUE_FOR_RCA,CELL_TAG,MAT_BALLOON_IND,maturity_conditional_flags,NOT_APPLICABLE,before_contractual_maturity,1200,1200,100.0%,PD001_01;PD001_02;PD001_03;PD001_04;PD001_05,A value is populated where the KB says the field is not applicable for this row context.,"Raise a data-treatment issue. Determine whether the value is a placeholder, mapping leakage, or a valid policy exception before modelling use.",True,resolved_cell_tags
3,HIGH,RAISE_ISSUE_FOR_RCA,CELL_TAG,NOI,income_measure,STALE_FROZEN,declared_field_sentinel,3,3,0.2%,PD029_06;PD058_06;PD087_06,A declared unavailable/sentinel value is present in a field expected to carry analytical information.,"Raise a source-data issue. Confirm the sentinel mapping, identify why the value was unavailable, and assess affected downstream use.",True,resolved_cell_tags
4,HIGH,RAISE_ISSUE_FOR_RCA,CELL_TAG,PRIN_PAYDOWN_AMT,amortisation_fields,NOT_APPLICABLE,non_amortising_contract,300,300,25.0%,PD004_01;PD004_02;PD004_03;PD004_04;PD004_05,A value is populated where the KB says the field is not applicable for this row context.,"Raise a data-treatment issue. Determine whether the value is a placeholder, mapping leakage, or a valid policy exception before modelling use.",True,resolved_cell_tags
5,MEDIUM,REVIEW_CONFIGURATION,UNSCOPED_RULE,BAL_AMT,drawn_balance,,exposure_change_indicators;behavioural_balance_update_frequency,0,0,nan%,,An applicable KB entry was not executed because a role binding or runtime declaration is missing.,Supply or review the listed prerequisite. Do not interpret the absence of tags from this route as a pass.,False,execution_plan:t2_d08_valsim_stale_frozen_ead_drawn_balance
6,MEDIUM,REVIEW_CONFIGURATION,UNSCOPED_RULE,BRR,internal_grade,,lower_frequency_fields_and_review_cycles,0,0,nan%,,An applicable KB entry was not executed because a role binding or runtime declaration is missing.,Supply or review the listed prerequisite. Do not interpret the absence of tags from this route as a pass.,False,execution_plan:t2_d08_valsim_stale_frozen_internal_grade
7,MEDIUM,REVIEW_CONFIGURATION,UNSCOPED_RULE,CURR_DPD,arrears_measure,,non_arrears_trigger,0,0,nan%,,An applicable KB entry was not executed because a role binding or runtime declaration is missing.,Supply or review the listed prerequisite. Do not interpret the absence of tags from this route as a pass.,False,execution_plan:t2_d08_valsim_not_applicable_lgd_arrears
8,INFO,APPLY_EXPECTED_TREATMENT,CELL_TAG,DFLT_12M,forward_labels,CENSORED,incomplete_forward_window,400,400,33.3%,PD001_09;PD001_10;PD001_11;PD001_12;PD002_09,The outcome is not observable within the declared data window or event state.,Apply the CENSORED treatment in downstream analysis; raise an RCA only if the declared observation boundary or event state is incorrect.,False,resolved_cell_tags
9,INFO,APPLY_EXPECTED_TREATMENT,CELL_TAG,SPREAD_BPS,spread_over_benchmark,NOT_APPLICABLE,fixed_rate_no_benchmark,396,396,33.0%,PD003_01

## 5. RCA evidence and configuration gaps

In [9]:
display(Markdown('### Findings where an issue and RCA are warranted'))
rca = actions['rca_evidence']
display(rca[['input_variable', 'matched_role', 'tag', 'reason_code', 'affected_cells', 'affected_share', 'sample_row_references', 'issue_statement', 'recommended_next_step']] if not rca.empty else pd.DataFrame({'outcome': ['No RCA-triggering evidence identified']}))
display(Markdown('### Coverage/configuration review'))
configuration = queue.loc[queue['user_decision'].eq('REVIEW_CONFIGURATION')] if not queue.empty else queue
display(configuration[['input_variable', 'scope', 'reason_code', 'issue_statement', 'recommended_next_step']] if not configuration.empty else pd.DataFrame({'outcome': ['No configuration gaps identified']}))

### Findings where an issue and RCA are warranted

,input_variable,matched_role,tag,reason_code,affected_cells,affected_share,sample_row_references,issue_statement,recommended_next_step
0,DEFAULT_FLAG,construction_constants,NOT_APPLICABLE,panel_scope_exclusion,1200,1.0000,PD001_01;PD001_02;PD001_03;PD001_04;PD001_05,A value is populated where the KB says the field is not applicable for this row context.,"Raise a data-treatment issue. Determine whether the value is a placeholder, mapping leakage, or a valid policy exception before modellin..."
1,INT_RT,priced_rate,STALE_FROZEN,zero_variance_at_declared_grain,51,0.0425,PD001_05;PD001_06;PD001_07;PD005_05;PD005_06,The value did not change at the KB-declared monitoring grain despite sufficient observations.,Raise an RCA on the source refresh and transformation path; confirm whether the field is genuinely stable or its updates are frozen.
2,MAT_BALLOON_IND,maturity_conditional_flags,NOT_APPLICABLE,before_contractual_maturity,1200,1.0000,PD001_01;PD001_02;PD001_03;PD001_04;PD001_05,A value is populated where the KB says the field is not applicable for this row context.,"Raise a data-treatment issue. Determine whether the value is a placeholder, mapping leakage, or a valid policy exception before modellin..."
3,NOI,income_measure,STALE_FROZEN,declared_field_sentinel,3,0.0025,PD029_06;PD058_06;PD087_06,A declared unavailable/sentinel value is present in a field expected to carry analytical information.,"Raise a source-data issue. Confirm the sentinel mapping, identify why the value was unavailable, and assess affected downstream use."
4,PRIN_PAYDOWN_AMT,amortisation_fields,NOT_APPLICABLE,non_amortising_contract,300,0.2500,PD004_01;PD004_02;PD004_03;PD004_04;PD004_05,A value is populated where the KB says the field is not applicable for this row context.,"Raise a data-treatment issue. Determine whether the value is a placeholder, mapping leakage, or a valid policy exception before modellin..."


### Coverage/configuration review

,input_variable,scope,reason_code,issue_statement,recommended_next_step
5,BAL_AMT,UNSCOPED_RULE,exposure_change_indicators;behavioural_balance_update_frequency,An applicable KB entry was not executed because a role binding or runtime declaration is missing.,Supply or review the listed prerequisite. Do not interpret the absence of tags from this route as a pass.
6,BRR,UNSCOPED_RULE,lower_frequency_fields_and_review_cycles,An applicable KB entry was not executed because a role binding or runtime declaration is missing.,Supply or review the listed prerequisite. Do not interpret the absence of tags from this route as a pass.
7,CURR_DPD,UNSCOPED_RULE,non_arrears_trigger,An applicable KB entry was not executed because a role binding or runtime declaration is missing.,Supply or review the listed prerequisite. Do not interpret the absence of tags from this route as a pass.


## 6. Expected treatment and expected-path evidence

In [10]:
expected_treatment = queue.loc[queue['user_decision'].eq('APPLY_EXPECTED_TREATMENT')] if not queue.empty else queue
display(Markdown('### Tags that follow declared semantics'))
display(expected_treatment[['input_variable', 'tag', 'reason_code', 'affected_cells', 'affected_share', 'recommended_next_step']] if not expected_treatment.empty else pd.DataFrame({'outcome': ['No expected-treatment tags identified']}))
display(Markdown('### Assessed fields with no triggered condition'))
display(actions['expected_path_summary'] if not actions['expected_path_summary'].empty else pd.DataFrame({'outcome': ['No completely clear assessed field identified']}))

### Tags that follow declared semantics

,input_variable,tag,reason_code,affected_cells,affected_share,recommended_next_step
8,DFLT_12M,CENSORED,incomplete_forward_window,400,0.333333,Apply the CENSORED treatment in downstream analysis; raise an RCA only if the declared observation boundary or event state is incorrect.
9,SPREAD_BPS,NOT_APPLICABLE,fixed_rate_no_benchmark,396,0.330000,Retain the NOT_APPLICABLE classification and exclude the cell from ordinary missing-value or model-feature treatment.


### Assessed fields with no triggered condition

,input_variable,semantic_roles,assessed_cells,no_tag_cells,user_decision,reason
0,CURR_DPD,arrears_measure,1200,1200,NO_ACTION_EXPECTED_PATH,The field was assessed and no KB condition was met.
1,OCC_PCT,utilisation_measure,1200,1200,NO_ACTION_EXPECTED_PATH,The field was assessed and no KB condition was met.


## 7. Evidence drill-down

The action queue is an aggregate. These tables retain traceability to individual cells, applied KB entries, suppressed competing tags, and unscoped prerequisites.

In [11]:
display(Markdown('### Resolved tagged cells'))
display(execution.resolved_cell_tags.head(50))
display(Markdown('### Rule execution coverage'))
display(execution.execution_plan)
display(Markdown('### Dataset diagnostic counts'))
display(summaries['dataset_summary'])

### Resolved tagged cells

,row_reference,input_variable,matched_role,rule,entry,tag,reason_code,input_value,is_populated,suppressed_tags,claim_count
0,PD001_01,DEFAULT_FLAG,construction_constants,t2_d08_valsim_rule_not_applicable,t2_d08_valsim_not_applicable_pd_construction_constant,NOT_APPLICABLE,panel_scope_exclusion,0.00,True,,1
1,PD001_01,MAT_BALLOON_IND,maturity_conditional_flags,t2_d08_valsim_rule_not_applicable,t2_d08_valsim_not_applicable_pd_before_maturity,NOT_APPLICABLE,before_contractual_maturity,0.00,True,,1
2,PD001_02,DEFAULT_FLAG,construction_constants,t2_d08_valsim_rule_not_applicable,t2_d08_valsim_not_applicable_pd_construction_constant,NOT_APPLICABLE,panel_scope_exclusion,0.00,True,,1
3,PD001_02,MAT_BALLOON_IND,maturity_conditional_flags,t2_d08_valsim_rule_not_applicable,t2_d08_valsim_not_applicable_pd_before_maturity,NOT_APPLICABLE,before_contractual_maturity,0.00,True,,1
4,PD001_03,DEFAULT_FLAG,construction_constants,t2_d08_valsim_rule_not_applicable,t2_d08_valsim_not_applicable_pd_construction_constant,NOT_APPLICABLE,panel_scope_exclusion,0.00,True,,1
5,PD001_03,MAT_BALLOON_IND,maturity_conditional_flags,t2_d08_valsim_rule_not_applicable,t2_d08_valsim_not_applicable_pd_before_maturity,NOT_APPLICABLE,before_contractual_maturity,0.00,True,,1
6,PD001_04,DEFAULT_FLAG,construction_constants,t2_d08_valsim_rule_not_applicable,t2_d08_valsim_not_applicable_pd_construction_constant,NOT_APPLICABLE,panel_scope_exclusion,0.00,True,,1
7,PD001_04,MAT_BALLOON_IND,maturity_conditional_flags,t2_d08_valsim_rule_not_applicable,t2_d08_valsim_not_applicable_pd_before_maturity,NOT_APPLICABLE,before_contractual_maturity,0.00,True,,1
8,PD001_05,DEFAULT_FLAG,construction_constants,t2_d08_valsim_rule_not_applicable,t2_d08_valsim_not_applicable_pd_construction_constant,NOT_APPLICABLE,panel_scope_exclusion,0.00,True,,1
9,PD001_05,INT_RT,priced_rate,t2_d08_valsim_rule_stale_frozen,t2_d08_valsim_stale_frozen_tracking_rate,STALE_FROZEN,zero_variance_at_declared_grain,4.25,True,,1


### Rule execution coverage

,input_variable,matched_role,rule,entry,assessment_grain,routing_status,missing_roles,missing_declarations
0,DFLT_12M,forward_labels,t2_d08_valsim_rule_censored,t2_d08_valsim_censored_pd_incomplete_forward_window,row_and_cell,READY_FOR_EVALUATION,,
1,CURR_DPD,arrears_measure,t2_d08_valsim_rule_stale_frozen,t2_d08_valsim_stale_frozen_arrears,segment_and_period,READY_FOR_EVALUATION,,
2,INT_RT,priced_rate,t2_d08_valsim_rule_stale_frozen,t2_d08_valsim_stale_frozen_tracking_rate,segment_and_period,READY_FOR_EVALUATION,,
3,BRR,internal_grade,t2_d08_valsim_rule_stale_frozen,t2_d08_valsim_stale_frozen_internal_grade,segment_and_declared_review_cycle,UNSCOPED,,lower_frequency_fields_and_review_cycles
4,NOI,income_measure,t2_d08_valsim_rule_stale_frozen,t2_d08_valsim_stale_frozen_income,segment_and_period,READY_FOR_EVALUATION,,
5,OCC_PCT,utilisation_measure,t2_d08_valsim_rule_stale_frozen,t2_d08_valsim_stale_frozen_utilisation,segment_and_period,READY_FOR_EVALUATION,,
6,BAL_AMT,drawn_balance,t2_d08_valsim_rule_stale_frozen,t2_d08_valsim_stale_frozen_ead_drawn_balance,entity_and_declared_update_cycle,UNSCOPED,exposure_change_indicators,behavioural_balance_update_frequency
7,CURR_DPD,arrears_measure,t2_d08_valsim_rule_not_applicable,t2_d08_valsim_not_applicable_lgd_arrears,row_and_cell,UNSCOPED,non_arrears_trigger,
8,DEFAULT_FLAG,construction_constants,t2_d08_valsim_rule_not_applicable,t2_d08_valsim_not_applicable_pd_construction_constant,row_and_cell,READY_FOR_EVALUATION,,
9,SPREAD_BPS,spread_over_benchmark,t2_d08_valsim_rule_not_applicable,t2_d08_valsim_not_applicable_pd_fixed_rate_spread,row_and_cell,READY_FOR_EVALUATION,,


### Dataset diagnostic counts

,input_rows,input_columns,columns_with_roles,examined_fields,unexamined_fields,ready_rule_routes,unscoped_rule_routes,rule_assessments,distinct_tagged_cells,censored_cells,stale_frozen_cells,not_applicable_cells,unclassified_assessments,unscoped_entries,multi_claim_cells,populated_where_not_applicable,run_verdict
0,1200,19,18,9,10,9,3,10800,3550,400,54,3096,0,3,0,2700,PARTIALLY_CLASSIFIED


## 8. Export the action-focused evidence pack

In [12]:
extra_frames = {
    **actions, 'column_binding_results': binding_results,
    **({'dictionary_evidence_comparison': dictionary_comparison} if COMPARE_DICTIONARY_MODES else {}),
}
output_path = export_execution(
    source_label, execution, summaries, OUTPUT_ROOT, extra_frames=extra_frames, file_version='v0_2',
    manifest_context={
        'notebook_version': '0.2', 'input_source': INPUT_SOURCE, 'data_rows': len(data),
        'snapshot_id': SNAPSHOT_ID if INPUT_SOURCE == 'snapshot' else None,
        'table': TABLE_NAME if INPUT_SOURCE == 'snapshot' else None,
        'aar_sourced_dictionary_used': USE_AAR_SOURCED_DICTIONARY if INPUT_SOURCE == 'snapshot' else None,
        'dictionary_used_for_matching': USE_DICTIONARY_FOR_MATCHING,
        'llm_fallback_enabled': USE_LLM_FALLBACK, 'binding_authority': binding_authority,
        'overall_user_decision': executive['overall_user_decision'],
    },
)
print('Action-focused evidence exported to:', output_path)

Action-focused evidence exported to: C:\Src\DataWorkbench\experiments\test-lab\t2_d08_Value_semantics\output\action_focused_v0_2\pd


## Interpretation guardrail

The diagnostic recommends an action; it does not establish business impact or root cause. A high affected-cell count describes scope, not severity. Escalation should preserve the listed row samples, KB reason, role binding, runtime declarations, and source lineage.